# 07 — Multi-Hop Reasoning Across Languages: The 0.512 → 0.252 → 0.135 Gradient
## تتبّع الاستدلال متعدد الخطوات عبر اللغات واللهجات · Tracing 2-Hop Concepts Across Layers

> **The Empirical Puzzle:** When testing a 2-hop factual query on **Gemma-2-2B** (Neuronpedia Circuit Tracer):
> 1. **English:** `"The capital of the country where the Suez Canal is located is"` → `Cairo` (**p = 0.512**)
> 2. **MSA:** `"عاصمة الدولة التي تقع فيها قناة السويس هي"` → `القاهرة` (**p = 0.252**)
> 3. **Egyptian (Masri):** `"عاصمة البلد اللي فيها قناة السويس هي"` → `القاهرة` (**p = 0.135**)
>
> All three variants arrive at the correct ground truth, but confidence drops by ~50% at each linguistic transition.

---

### Mechanistic Hypotheses to Test:
1. **Tokenization Fragmentation:** How severely does subword tokenization split the subject (`Suez Canal` vs `قناة السويس`) and the relation clitics (`where ... is located` vs `التي تقع فيها` vs `اللي فيها`)?
2. **The Intermediate Hop (Hop 1):** Does the residual stream develop a strong latent representation of the intermediate entity (`Egypt` / `مصر`) at mid layers before generating the capital (`Cairo` / `القاهرة`)?
3. **The "English Pivot" Hypothesis:** In mid-layers, does the model translate Arabic queries into English conceptual directions, reason in English, and translate back to Arabic in late layers?
4. **Layer-Commitment Lag:** At which layer does the residual stream commit to the final answer across English vs MSA vs Masri?


In [1]:
from __future__ import annotations

import os
import torch
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Execution Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.5.1+rocm6.2
Execution Device: cuda
Device Name: AMD Radeon 8060S Graphics


In [2]:
# 1. Prompts for the 2-hop query across English, MSA, and Masri
PROMPTS = {
    "EN": "The capital of the country where the Suez Canal is located is",
    "MSA": "عاصمة الدولة التي تقع فيها قناة السويس هي",
    "Masri": "عاصمة البلد اللي فيها قناة السويس هي",
}

# 2. Intermediate Hop (Country) candidates to track in residual stream
HOP1_TARGETS = {
    "EN": ["Egypt", " Egypt"],
    "AR": ["مصر", " مصر"],
}

# 3. Final Hop (Capital) candidates
HOP2_TARGETS = {
    "EN": ["Cairo", " Cairo"],
    "AR": ["القاهرة", " القاهرة"],
}

print("=== Multi-Hop Reasoning Prompts ===")
for lang, p in PROMPTS.items():
    print(f"[{lang:<5}] {p}")

print("\n=== Target Candidates ===")
print(f"Hop 1 (Country): EN={HOP1_TARGETS['EN']}, AR={HOP1_TARGETS['AR']}")
print(f"Hop 2 (Capital): EN={HOP2_TARGETS['EN']}, AR={HOP2_TARGETS['AR']}")


=== Multi-Hop Reasoning Prompts ===
[EN   ] The capital of the country where the Suez Canal is located is
[MSA  ] عاصمة الدولة التي تقع فيها قناة السويس هي
[Masri] عاصمة البلد اللي فيها قناة السويس هي

=== Target Candidates ===
Hop 1 (Country): EN=['Egypt', ' Egypt'], AR=['مصر', ' مصر']
Hop 2 (Capital): EN=['Cairo', ' Cairo'], AR=['القاهرة', ' القاهرة']


In [3]:
from transformers import AutoTokenizer
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

print("=== Tokenization Footprints across Languages ===\n")
for lang, p in PROMPTS.items():
    tokens = tokenizer.tokenize(p)
    ids = tokenizer.encode(p, add_special_tokens=False)
    print(f"[{lang}] {len(tokens)} tokens:")
    formatted = " | ".join([f"{t}({i})" for t, i in zip(tokens, ids)])
    print(f"  {formatted}\n")

print("=== Target Token IDs (Hop 1 & Hop 2) ===")
targets_to_inspect = ["Cairo", " Cairo", "القاهرة", " القاهرة", "Egypt", " Egypt", "مصر", " مصر"]
target_rows = []
for text in targets_to_inspect:
    toks = tokenizer.tokenize(text)
    t_ids = tokenizer.encode(text, add_special_tokens=False)
    target_rows.append({
        "Surface String": repr(text),
        "Tokens": toks,
        "Token Count": len(toks),
        "Primary ID (First)": t_ids[0]
    })

df_targets = pd.DataFrame(target_rows)
print(df_targets.to_string(index=False))


=== Tokenization Footprints across Languages ===

[EN] 12 tokens:
  The(651) | ▁capital(6037) | ▁of(576) | ▁the(573) | ▁country(3170) | ▁where(1570) | ▁the(573) | ▁Suez(124399) | ▁Canal(31619) | ▁is(603) | ▁located(7023) | ▁is(603)

[MSA] 12 tokens:
  ع(235404) | اصمة(167892) | ▁الدولة(126786) | ▁التي(20146) | ▁ت(2069) | قع(21766) | ▁فيها(49256) | ▁قناة(222120) | ▁الس(9882) | و(235363) | يس(14748) | ▁هي(35235)

[Masri] 10 tokens:
  ع(235404) | اصمة(167892) | ▁البلد(163634) | ▁اللي(115867) | ▁فيها(49256) | ▁قناة(222120) | ▁الس(9882) | و(235363) | يس(14748) | ▁هي(35235)

=== Target Token IDs (Hop 1 & Hop 2) ===
Surface String      Tokens  Token Count  Primary ID (First)
       'Cairo'     [Cairo]            1              172382
      ' Cairo'    [▁Cairo]            1               53731
     'القاهرة' [الق, اهرة]            2               93554
    ' القاهرة'  [▁القاهرة]            1              211639
       'Egypt'     [Egypt]            1               77170
      ' Egypt'    [▁Egy

In [4]:
# Cell 5: Load Gemma-2-2B and run forward pass with activation cache
MODEL_NAME = "google/gemma-2-2b"
print(f"Loading {MODEL_NAME} onto {DEVICE}...")

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()

print(f"Model loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, d_vocab={model.cfg.d_vocab}\n")

prompt_data = {}
for lang, prompt_text in PROMPTS.items():
    tokens = model.to_tokens(prompt_text)
    with torch.no_grad():
        logits, cache = model.run_with_cache(tokens)
    
    prompt_data[lang] = {
        "tokens": tokens,
        "logits": logits,
        "cache": cache,
        "final_token_logits": logits[0, -1],
    }
    
    # Top-1 prediction and probability at the final prompt position
    probs = torch.softmax(logits[0, -1], dim=-1)
    top1_id = torch.argmax(probs).item()
    top1_prob = probs[top1_id].item()
    top1_str = model.to_string(top1_id)
    print(f"[{lang:<5}] Top-1 output: {top1_str!r} (p={top1_prob:.3f})")


Loading google/gemma-2-2b onto cuda...
Loaded pretrained model google/gemma-2-2b into HookedTransformer
Model loaded: 26 layers, d_model=2304, d_vocab=256000

[EN   ] Top-1 output: ' Cairo' (p=0.243)
[MSA  ] Top-1 output: ' القاهرة' (p=0.241)
[Masri] Top-1 output: ' القاهرة' (p=0.134)
